# Recursive flop and preflop outcomes

The chance tree is defined from the bottom up: `turn_outcomes` evaluates 44 rivers, `flop_outcomes` aggregates 45 labelled turn children, and `preflop_outcomes` aggregates 17,296 unordered flop children.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from cards import make_hand
from flop import flop_outcomes
from preflop import flop_children, preflop_flop_outcomes, preflop_outcomes
from showdown import INCOMPATIBLE

## Flop: 45 turns × 44 rivers

Every row below is a labelled turn child whose counts come from `turn_outcomes`.

In [2]:
flop = make_hand(['2h', '7h', 'Tc'])
first = make_hand(['Ah', 'Kh'])
second = make_hand(['Js', 'Jd'])
flop_result = flop_outcomes(flop, first, second)

if flop_result == INCOMPATIBLE:
    print('Incompatible flop and private hands')
else:
    print(f'Runouts: {flop_result.total:,}')
    print(f'W/L/T: {flop_result.wins}/{flop_result.losses}/{flop_result.ties}')
    print(f'Exact equity: {flop_result.equity} = {float(flop_result.equity):.2%}')

Runouts: 1,980
W/L/T: 1078/902/0
Exact equity: 49/90 = 54.44%


In [3]:
if flop_result == INCOMPATIBLE:
    turns = pd.DataFrame([{'outcome': 'incompatible'}])
else:
    turns = pd.DataFrame([
        {
            'turn': child.turn_name,
            'W': child.result.wins,
            'L': child.result.losses,
            'T': child.result.ties,
            'equity': float(child.result.equity),
        }
        for child in flop_result.turns
    ])
turns

,turn,W,L,T,equity
0,2c,14,30,0,0.318182
1,3c,15,29,0,0.340909
2,4c,15,29,0,0.340909
3,5c,15,29,0,0.340909
4,6c,15,29,0,0.340909
5,7c,14,30,0,0.318182
6,8c,15,29,0,0.340909
7,9c,15,29,0,0.340909
8,Jc,10,34,0,0.227273
9,Qc,16,28,0,0.363636


## Preflop: lazy recursive traversal

Two fixed hands leave 17,296 possible unordered flops. A complete exact traversal represents 34,246,080 ordered turn-river runouts, so `preflop_flop_outcomes` yields one recursively calculated flop summary at a time. `preflop_outcomes(first, second)` consumes the entire iterator when the full aggregate is wanted.

In [4]:
possible_flops = tuple(flop_children(first, second))
print(f'Compatible flops: {len(possible_flops):,}')
print(f'Ordered turn-river runouts: {len(possible_flops) * 45 * 44:,}')

Compatible flops: 17,296
Ordered turn-river runouts: 34,246,080


In [5]:
# Inspect one recursively evaluated flop without starting the full traversal.
first_flop = next(preflop_flop_outcomes(first, second))
first_flop

FlopOutcome(flop=7, wins=670, losses=1168, ties=142)

To run the complete calculation, evaluate `preflop_result = preflop_outcomes(first, second)`. It is intentionally not executed in this notebook by default because the current exact Python evaluator must visit the full chance tree.